# Unified temporal robustness analysis

This report combines the visual temporal analysis of the legacy notebook with the leakage-safe family, matching, semantic, and cohort estimands of the current temporal comparison. It reads one explicitly pinned immutable parent and writes only a content-addressed derived enrichment.

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from artifact_storage import read_artifact
from temporal_unified_analysis import (
    UnifiedAnalysisConfig, normalize_first_finite,
    validate_completed_parent,
)
from temporal_unified_enrichment import build_unified_enrichment, load_enrichment
from temporal_cri import CRIAnalysisConfig, build_cri_analysis, load_cri_analysis

# All analysis choices are intentionally centralized here.
ARTIFACT_ROOT = Path('stats/temporal_robustness')
PARENT_HASH = '5fd57eb7b61700cda81e'
COSINE_THRESHOLD = 0.60
OVERLAP_THRESHOLD = 0.70
OVERLAP_PERCENTILE = 70
RECURRENCE_MIN = 0.50  # strict > comparison
ELIGIBILITY_TARGET = 0.50
ELIGIBILITY_MINIMUM_T0_ACTIVE = 10
ACTIVATION_TARGETS = (0.10, 0.30, 0.50)  # P90, P70, P50; never pooled
COHORTS = ('all_comer', 'pipeline_unseen')
TCAV_REPETITIONS = 15
TCAV_FDR_ALPHA = 0.05
TCAV_NEUTRAL_BAND = (0.40, 0.60)
BOOTSTRAP_REPETITIONS = 1000
BOOTSTRAP_SEED = 42

CONFIG = UnifiedAnalysisConfig(
    parent_hash=PARENT_HASH, cosine_threshold=COSINE_THRESHOLD,
    overlap_threshold=OVERLAP_THRESHOLD, overlap_percentile=OVERLAP_PERCENTILE,
    recurrence_min=RECURRENCE_MIN,
    eligibility_activation_target=ELIGIBILITY_TARGET,
    eligibility_minimum_t0_active=ELIGIBILITY_MINIMUM_T0_ACTIVE,
    activation_targets=ACTIVATION_TARGETS, cohorts=COHORTS,
    tcav_repetitions=TCAV_REPETITIONS, tcav_fdr_alpha=TCAV_FDR_ALPHA,
    tcav_neutral_lower=TCAV_NEUTRAL_BAND[0],
    tcav_neutral_upper=TCAV_NEUTRAL_BAND[1],
    bootstrap_repetitions=BOOTSTRAP_REPETITIONS, bootstrap_seed=BOOTSTRAP_SEED,
)
TARGET_LABEL = {0.10: 'P90', 0.30: 'P70', 0.50: 'P50'}
VIEW_LABEL = {'cosine_qualified': 'Cosine-qualified', 'intersection': 'Consensus intersection'}
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)

### Factor-level TCAV scores

This audit table shows the actual TCAV result for every evaluated member factor, including factors that do not pass the significance and neutral-band filters. A row is specific to its reference/split, activation percentile, cohort, and temporal distance. `matching_views` records whether that same factor-level result belongs to the recurrent cosine-qualified view, the recurrent consensus intersection, or both.

## 1. Run identity and estimands

A **canonical factor family** is anchored by one factor from the canonical SAE seed for a reference-year/patient-split experiment. Its members are exact factor IDs selected independently in the other SAE retrainings. A geometric match is decoder-cosine-qualified; recurrence is the fraction of noncanonical SAE retrainings with a qualifying match. `intersection` additionally requires cosine and activation-overlap matching to select the same member factor.

Activation thresholds are frozen from reference-only data. P90, P70, and P50 therefore denote separate estimands. `pipeline_unseen` excludes patients used to fit the reference pipeline while retaining returning T0 patients, genuinely new entrants, and prior nonreference returners.

In [ ]:
# This cell refuses partial or actively changing parents before doing any work.
PARENT_MANIFEST_PATH, PARENT = validate_completed_parent(ARTIFACT_ROOT, CONFIG)
ENRICHMENT_MANIFEST_PATH = build_unified_enrichment(ARTIFACT_ROOT, CONFIG)
ENRICHMENT_MANIFEST = json.loads(ENRICHMENT_MANIFEST_PATH.read_text())
DERIVED = {name: pd.DataFrame(rows) for name, rows in load_enrichment(ENRICHMENT_MANIFEST_PATH).items()}
CRI_MANIFEST_PATH = build_cri_analysis(ENRICHMENT_MANIFEST_PATH, CONFIG, CRIAnalysisConfig())
CRI = {name: pd.DataFrame(rows) for name, rows in load_cri_analysis(CRI_MANIFEST_PATH).items()}

def load_parent_table(name):
    descriptor = PARENT.get('aggregate_artifacts', {}).get(name)
    return pd.DataFrame() if descriptor is None else pd.DataFrame(read_artifact(PARENT_MANIFEST_PATH.parent, descriptor))

PARENT_TABLES = {name: load_parent_table(name) for name in (
    'matching_recurrence', 'threshold_membership', 'factor_families', 'rules',
    'cavs', 'semantic_selection_diagnostics', 'semantic_selection_summary',
    'lead_lag', 'change_points',
)}
identity = pd.DataFrame([{
    'parent_hash': PARENT_HASH,
    'parent_manifest_sha256': ENRICHMENT_MANIFEST['parent_manifest_sha256'],
    'enrichment_hash': ENRICHMENT_MANIFEST['enrichment_hash'],
    'reference_years': PARENT['config']['reference_years'],
    'patient_split_seeds': PARENT['config']['patient_split_seeds'],
    'sae_seeds': PARENT['config']['sae_seeds'],
    'successful_experiments': len(PARENT['successful_experiments']),
    'failed_experiments': len(PARENT['failed_experiments']),
    'source_fingerprints': PARENT['source_fingerprints'],
    'cohorts': COHORTS, 'activation_targets': [TARGET_LABEL[x] for x in ACTIVATION_TARGETS],
}])
display(identity.T.rename(columns={0: 'value'}))
display(Markdown(f"**Estimand:** {PARENT['estimand']}"))

## 2. Family universe and selection funnel

The ladder is nested. **Matched** means at least one headline cosine match and is shown only here; it is not a standalone temporal-analysis cohort. CAV-ready and TCAV-valid are badges, not robustness requirements.

In [ ]:
ladder = DERIVED['family_ladder'].copy()
stages = ['eligible', 'matched', 'recurrent', 'consensus', 'interpretable', 'dual_source']
stage_counts = pd.DataFrame({
    'stage': stages,
    'families': [int(ladder[stage].sum()) for stage in stages],
})
stage_counts['lost_from_previous'] = stage_counts['families'].shift(fill_value=len(ladder)) - stage_counts['families']
display(stage_counts)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(stage_counts['stage'], stage_counts['families'], color=plt.cm.Blues(np.linspace(.35, .9, len(stages))))
for index, row in stage_counts.iterrows():
    ax.text(index, row['families'], f"{row['families']:,}", ha='center', va='bottom')
ax.set(title='Nested family robustness ladder', ylabel='Canonical factor families')
ax.tick_params(axis='x', rotation=25)
plt.show()

badges = ladder.groupby(['cav_ready', 'tcav_valid'], dropna=False).size().rename('families').reset_index()
display(badges)
signature_columns = ['recurrent', 'consensus', 'interpretable', 'dual_source', 'cav_ready', 'tcav_valid']
overlap = (ladder.groupby(signature_columns, dropna=False).size().rename('families').reset_index()
           .sort_values('families', ascending=False).head(15))
overlap['signature'] = overlap.apply(lambda row: ' ∩ '.join(name for name in signature_columns if row[name]) or 'none', axis=1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(overlap['signature'][::-1], overlap['families'][::-1], color='#4c78a8')
ax.set(title='Largest family-set intersections (UpSet-style)', xlabel='Families')
plt.show()

## 3. Matching robustness

The headline views use cosine 0.60, overlap 0.70, P70 overlap masks, and recurrence strictly greater than 0.50. The sensitivity heatmaps below remain descriptive and do not retune that headline configuration.

In [ ]:
recurrence = PARENT_TABLES['matching_recurrence'].copy()
recurrence['recurrent_strict'] = pd.to_numeric(recurrence['recurrence'], errors='coerce').gt(RECURRENCE_MIN)
cosine_grid = (recurrence[recurrence['matching_view'].eq('cosine_qualified')]
               .groupby('cosine_threshold', as_index=False)['recurrent_strict'].sum())
intersection_grid = (recurrence[recurrence['matching_view'].eq('intersection') & recurrence['recurrent_strict']]
    .groupby(['cosine_threshold', 'overlap_percentile', 'overlap_threshold'])['factor_family_uid'].nunique().rename('recurrent_families').reset_index())
fig, axis = plt.subplots(figsize=(6.5, 4.5))
axis.plot(cosine_grid['cosine_threshold'], cosine_grid['recurrent_strict'], marker='o')
axis.axvline(COSINE_THRESHOLD, color='black', linestyle='--', alpha=.5)
axis.set(title='Cosine threshold sensitivity', xlabel='Cosine threshold', ylabel='Recurrent families')
fig.tight_layout(); plt.show()
cosine_thresholds = sorted(intersection_grid['cosine_threshold'].dropna().unique())
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
vmax = intersection_grid['recurrent_families'].max()
for axis, threshold in zip(axes.flat, cosine_thresholds):
    pivot = (intersection_grid[intersection_grid['cosine_threshold'].eq(threshold)]
        .pivot(index='overlap_percentile', columns='overlap_threshold', values='recurrent_families')
        .reindex(index=sorted(intersection_grid['overlap_percentile'].unique()), columns=sorted(intersection_grid['overlap_threshold'].unique())))
    image = axis.imshow(pivot, aspect='auto', cmap='Blues', vmin=0, vmax=vmax)
    axis.set(xticks=range(len(pivot.columns)), xticklabels=pivot.columns, yticks=range(len(pivot.index)), yticklabels=[f'P{x}' for x in pivot.index], xlabel='Overlap threshold', ylabel='Overlap mask', title=f'Intersection sensitivity · cosine ≥ {threshold:.1f}')
    for y in range(len(pivot.index)):
        for x in range(len(pivot.columns)):
            value = pivot.iloc[y, x]
            axis.text(x, y, '' if pd.isna(value) else int(value), ha='center', va='center')
fig.colorbar(image, ax=axes.ravel().tolist(), label='Unique recurrent families')
fig.tight_layout(); plt.show()

headline = recurrence[
    recurrence['cosine_threshold'].eq(COSINE_THRESHOLD)
    & recurrence['matching_view'].isin(['cosine_qualified', 'intersection'])
].copy()
headline = headline[(headline['matching_view'].eq('cosine_qualified')) | (
    headline['overlap_percentile'].eq(OVERLAP_PERCENTILE) & headline['overlap_threshold'].eq(OVERLAP_THRESHOLD)
)]
diagnostic = (headline.pivot_table(index='factor_family_uid', columns='matching_view', values='recurrent_strict', aggfunc='max', fill_value=False).reset_index())
diagnostic['membership'] = np.select([
    diagnostic.get('cosine_qualified', False) & diagnostic.get('intersection', False),
    diagnostic.get('cosine_qualified', False), diagnostic.get('intersection', False),
], ['both', 'cosine only', 'intersection only'], default='neither')
display(diagnostic['membership'].value_counts().rename_axis('configuration membership').reset_index(name='families'))

## 4. Rule quality and interpretability

Semantic rules are the headline explanation source. High-precision rules remain visible in secondary panels on exactly the same axes; they are not called primary and do not replace the semantic estimand.

In [ ]:
rules = PARENT_TABLES['rules'].copy()
rules = rules[rules['activation_target'].isin(ACTIVATION_TARGETS)]
quality = ['precision', 'recall', 'f2', 'lift']
fig, axes = plt.subplots(2, len(quality), figsize=(16, 7), sharex='col')
for row_index, source in enumerate(('semantic', 'high_precision')):
    selected = rules[rules['rule_source'].eq(source)]
    for axis, metric in zip(axes[row_index], quality):
        values = [pd.to_numeric(selected[selected['activation_target'].eq(target)][metric], errors='coerce').dropna() for target in ACTIVATION_TARGETS]
        axis.boxplot(values, labels=[TARGET_LABEL[target] for target in ACTIVATION_TARGETS], showfliers=False)
        axis.set_title(metric.replace('_', ' ').title())
        if metric != 'lift': axis.set_ylim(0, 1)
        if axis is axes[row_index, 0]: axis.set_ylabel(source.replace('_', ' ').title())
fig.suptitle('Rule-quality distributions on equal source-specific panels')
fig.tight_layout(); plt.show()
rule_summary = (rules.groupby(['rule_source', 'activation_target'], as_index=False)
    .agg(attempted=('factor_family_uid', 'size'), valid=('valid', 'sum'), families=('factor_family_uid', 'nunique'), precision=('precision', 'mean'), recall=('recall', 'mean'), f2=('f2', 'mean')))
rule_summary['activation_percentile'] = rule_summary['activation_target'].map(TARGET_LABEL)
display(rule_summary)
display(PARENT_TABLES['semantic_selection_summary'].head(30))

In [ ]:
valid_rules = rules[rules['valid'].eq(True)].copy()
fig, axes = plt.subplots(2, len(quality), figsize=(16, 7), sharex='col')
for row_index, source in enumerate(('semantic', 'high_precision')):
    selected = valid_rules[valid_rules['rule_source'].eq(source)]
    counts = [int(selected['activation_target'].eq(target).sum()) for target in ACTIVATION_TARGETS]
    labels = [f"{TARGET_LABEL[target]}\n(n={count})" for target, count in zip(ACTIVATION_TARGETS, counts)]
    for axis, metric in zip(axes[row_index], quality):
        values = [pd.to_numeric(selected[selected['activation_target'].eq(target)][metric], errors='coerce').dropna() for target in ACTIVATION_TARGETS]
        axis.boxplot(values, labels=labels, showfliers=False)
        axis.set_title(metric.replace('_', ' ').title())
        if metric != 'lift': axis.set_ylim(0, 1)
        if axis is axes[row_index, 0]: axis.set_ylabel(source.replace('_', ' ').title())
fig.suptitle('Rule-quality distributions among valid rules only')
fig.tight_layout(); plt.show()
display(valid_rules.groupby(['rule_source', 'activation_target'], as_index=False).agg(valid_rules=('factor_family_uid', 'size'), families=('factor_family_uid', 'nunique'), precision=('precision', 'mean'), recall=('recall', 'mean'), f2=('f2', 'mean')))

## 5. Temporal robustness and model performance

Ribbons are 95% hierarchical bootstrap intervals, not standard deviations. Each figure fixes cohort, matching view, rule source, and activation percentile before aggregation. TCAV is shown only in its actual 0–1 score units. Model-performance curves always come from the original TabPFN system whose reference experiment supplies the concept-robustness measurements; zero death F1 values are retained as valid primary outcomes.

In [ ]:
performance = DERIVED['primary_performance'].copy()
factors = DERIVED['headline_factor_metrics'].copy()
tcav_all = DERIVED['tcav_significance'].copy()
tcav = tcav_all[tcav_all['tcav_valid'].eq(True)].copy() if not tcav_all.empty else tcav_all
status = DERIVED['status_composition'].copy()

def hierarchical_trajectory(frame, metric, group_fields, unit_fields):
    rows = []
    for key, group in frame.groupby(list(group_fields), dropna=False):
        key = key if isinstance(key, tuple) else (key,)
        unit = group.groupby(list(unit_fields), dropna=False, as_index=False)[metric].mean().dropna(subset=[metric])
        if unit.empty: continue
        cluster_fields = [field for field in ('reference_year', 'patient_split_seed') if field in unit_fields]
        clusters = [rows[metric].to_numpy(float) for _, rows in unit.groupby(cluster_fields, dropna=False)]
        digest = hashlib.sha256(repr((metric, key)).encode()).hexdigest()
        seed = BOOTSTRAP_SEED + int(digest[:8], 16) % 100000
        rng = np.random.default_rng(seed)
        boot = []
        for _ in range(BOOTSTRAP_REPETITIONS):
            sampled_clusters = rng.integers(0, len(clusters), len(clusters))
            cluster_means = []
            for position in sampled_clusters:
                values = clusters[position]
                cluster_means.append(np.mean(values[rng.integers(0, len(values), len(values))]))
            boot.append(np.mean(cluster_means))
        low, high = np.percentile(boot, [2.5, 97.5])
        rows.append({**dict(zip(group_fields, key)), 'metric': metric, 'mean': float(np.mean([np.mean(values) for values in clusters])), 'lower_95': low, 'upper_95': high, 'support': len(unit)})
    return pd.DataFrame(rows)

performance_summary = pd.concat([hierarchical_trajectory(
    performance, metric, ('cohort_view', 'temporal_distance'), ('reference_year', 'patient_split_seed')
) for metric in ('macro_f1', 'death_f1')], ignore_index=True)
factor_summaries = []
for metric in ('f2', 'prevalence_ratio', 'jaccard', 'feature_association_cosine', 'activation_magnitude', 'activation_magnitude_ratio'):
    factor_summaries.append(hierarchical_trajectory(
        factors, metric, ('cohort_view', 'matching_view', 'rule_source', 'activation_target', 'temporal_distance'),
        ('reference_year', 'patient_split_seed', 'factor_family_uid', 'member_sae_seed', 'member_factor_id'),
    ))
factor_summary = pd.concat(factor_summaries, ignore_index=True)
tcav_summary = (pd.DataFrame() if tcav.empty else hierarchical_trajectory(
    tcav, 'tcav', ('cohort_view', 'matching_view', 'rule_source', 'activation_target', 'temporal_distance'),
    ('reference_year', 'patient_split_seed', 'factor_family_uid', 'member_sae_seed', 'member_factor_id'),
))

def ribbon(axis, rows, label, color=None):
    rows = rows.sort_values('temporal_distance')
    if rows.empty: return
    line = axis.plot(rows['temporal_distance'], rows['mean'], marker='o', label=label, color=color)[0]
    axis.fill_between(rows['temporal_distance'], rows['lower_95'], rows['upper_95'], color=line.get_color(), alpha=.18)

def aligned_native_panels(cohort, view, source):
    metrics = [('macro_f1', 'Macro F1'), ('death_f1', 'Death F1'), ('f2', 'Rule F2'), ('prevalence_ratio', 'Prevalence ratio'), ('jaccard', 'Jaccard'), ('feature_association_cosine', 'Feature-association cosine'), ('activation_magnitude', 'Activation magnitude'), ('activation_magnitude_ratio', 'Magnitude ratio'), ('tcav', 'Actual TCAV (valid)')]
    fig, axes = plt.subplots(len(ACTIVATION_TARGETS), len(metrics), figsize=(27, 10), sharex=True)
    for row_index, target in enumerate(ACTIVATION_TARGETS):
        for axis, (metric, label) in zip(axes[row_index], metrics):
            if metric in {'macro_f1', 'death_f1'}:
                selected = performance_summary[performance_summary['cohort_view'].eq(cohort) & performance_summary['metric'].eq(metric)]
            elif metric == 'tcav':
                selected = tcav_summary[(tcav_summary.get('cohort_view') == cohort) & (tcav_summary.get('matching_view') == view) & (tcav_summary.get('rule_source') == source) & (tcav_summary.get('activation_target') == target)] if not tcav_summary.empty else pd.DataFrame()
            else:
                selected = factor_summary[(factor_summary['cohort_view'].eq(cohort)) & (factor_summary['matching_view'].eq(view)) & (factor_summary['rule_source'].eq(source)) & (factor_summary['activation_target'].eq(target)) & (factor_summary['metric'].eq(metric))]
            ribbon(axis, selected, TARGET_LABEL[target])
            axis.set_title(label)
            if metric in {'macro_f1', 'death_f1', 'f2', 'jaccard', 'tcav'}: axis.set_ylim(0, 1)
            if metric == 'feature_association_cosine': axis.set_ylim(-1, 1)
            if metric in {'prevalence_ratio', 'activation_magnitude_ratio'}: axis.axhline(1, color='black', linewidth=1, alpha=.35)
            if row_index == len(ACTIVATION_TARGETS)-1: axis.set_xlabel('Temporal distance')
        axes[row_index, 0].set_ylabel(TARGET_LABEL[target])
    fig.suptitle(f"{cohort.replace('_', ' ')} · {VIEW_LABEL[view]} · {source.replace('_', ' ')}", y=1.01)
    fig.tight_layout(); plt.show()

for cohort in COHORTS:
    for view in ('cosine_qualified', 'intersection'):
        aligned_native_panels(cohort, view, 'semantic')
        aligned_native_panels(cohort, view, 'high_precision')

### Normalized overview

Each curve is divided by its own first finite measurement. Pipeline-unseen therefore begins at its first available future distance. Zero baselines remain unavailable and are not silently replaced. TCAV is deliberately excluded.

In [ ]:
def normalized_curve(summary, metric, filters):
    selected = summary[summary['metric'].eq(metric)].copy()
    for field, value in filters.items(): selected = selected[selected[field].eq(value)]
    rows = normalize_first_finite(selected.to_dict('records'), metric='mean', group_fields=tuple(filters))
    return pd.DataFrame(rows)

normalized_metrics = [('macro_f1', 'Macro F1'), ('death_f1', 'Death F1'), ('f2', 'Rule F2'), ('prevalence_ratio', 'Prevalence ratio'), ('jaccard', 'Jaccard'), ('feature_association_cosine', 'Feature-association cosine'), ('activation_magnitude_ratio', 'Activation magnitude')]
for cohort in COHORTS:
    for view in ('cosine_qualified', 'intersection'):
        fig, axes = plt.subplots(1, len(ACTIVATION_TARGETS), figsize=(18, 4.5), sharey=True)
        for axis, target in zip(axes, ACTIVATION_TARGETS):
            for metric, label in normalized_metrics:
                if metric in {'macro_f1', 'death_f1'}:
                    curve = normalized_curve(performance_summary, metric, {'cohort_view': cohort})
                else:
                    curve = normalized_curve(factor_summary, metric, {'cohort_view': cohort, 'matching_view': view, 'rule_source': 'semantic', 'activation_target': target})
                if not curve.empty: axis.plot(curve['temporal_distance'], curve['normalized_value'], marker='o', label=label)
            axis.axhline(1, color='black', linewidth=1, alpha=.35)
            axis.set(title=TARGET_LABEL[target], xlabel='Temporal distance')
        axes[0].set_ylabel('Value / first finite value')
        axes[-1].legend(bbox_to_anchor=(1.03, 1), loc='upper left', fontsize=8)
        fig.suptitle(f"Normalized overview · {cohort.replace('_', ' ')} · {VIEW_LABEL[view]}")
        fig.tight_layout(); plt.show()

In [ ]:
tcav_factor_columns = [
    'reference_year', 'patient_split_seed', 'factor_family_uid',
    'member_sae_seed', 'member_factor_id', 'activation_percentile',
    'activation_target', 'rule_source', 'target_role', 'cohort_view',
    'test_year', 'temporal_distance', 'matching_views', 'tcav',
    'tcav_std', 'repetition_count', 'p_value', 'q_value', 'tcav_valid',
]
if tcav_all.empty:
    tcav_factor_table = pd.DataFrame(columns=tcav_factor_columns)
else:
    tcav_factor_table = tcav_all.copy()
    factor_result_keys = [
        'reference_year', 'patient_split_seed', 'factor_family_uid',
        'member_sae_seed', 'member_factor_id', 'activation_target',
        'rule_source', 'target_role', 'cohort_view', 'test_year',
        'temporal_distance',
    ]
    matching_membership = (tcav_factor_table
        .groupby(factor_result_keys, dropna=False)['matching_view']
        .agg(lambda values: ', '.join(sorted(set(values.dropna().astype(str)))))
        .rename('matching_views').reset_index())
    tcav_factor_table = (tcav_factor_table
        .drop(columns=['matching_view'], errors='ignore')
        .drop_duplicates(subset=factor_result_keys)
        .merge(matching_membership, on=factor_result_keys, how='left'))
    tcav_factor_table['activation_percentile'] = tcav_factor_table['activation_target'].map(TARGET_LABEL)
    tcav_factor_table = tcav_factor_table[tcav_factor_columns].sort_values(
        ['cohort_view', 'activation_target', 'reference_year',
         'patient_split_seed', 'factor_family_uid', 'member_sae_seed',
         'member_factor_id', 'temporal_distance'],
        na_position='last',
    ).reset_index(drop=True)
display(tcav_factor_table[(tcav_factor_table['tcav'] != 0) & (tcav_factor_table['tcav'] != 1)])

### Reference-year matrices and post-hoc temporal categories

Triangular matrices retain the reference-year estimand. Temporal categories are displayed only as outcomes and never used to select families.

In [ ]:
def triangular_heatmap(frame, metric, title, filters):
    selected = frame.copy()
    for field, value in filters.items(): selected = selected[selected[field].eq(value)]
    matrix = selected.pivot_table(index='reference_year', columns='test_year', values=metric, aggfunc='mean')
    if matrix.empty: return
    fig, ax = plt.subplots(figsize=(7, 5))
    image = ax.imshow(matrix, aspect='auto', cmap='viridis')
    ax.set(xticks=range(len(matrix.columns)), xticklabels=matrix.columns, yticks=range(len(matrix.index)), yticklabels=matrix.index, xlabel='Test year', ylabel='Reference year', title=title)
    fig.colorbar(image, ax=ax, label=metric.replace('_', ' ')); fig.tight_layout(); plt.show()

for target in ACTIVATION_TARGETS:
    triangular_heatmap(factors, 'prevalence_ratio', f"Prevalence ratio · {TARGET_LABEL[target]} · all comer · intersection", {'cohort_view': 'all_comer', 'matching_view': 'intersection', 'rule_source': 'semantic', 'activation_target': target})

for cohort in COHORTS:
    for target in ACTIVATION_TARGETS:
        selected = status[(status['cohort_view'].eq(cohort)) & (status['matching_view'].eq('intersection')) & (status['rule_source'].eq('semantic')) & (status['activation_target'].eq(target))]
        composition = selected.groupby(['temporal_distance', 'status'], as_index=False)['status_proportion'].mean()
        if composition.empty: continue
        pivot = composition.pivot(index='temporal_distance', columns='status', values='status_proportion').fillna(0)
        pivot.plot.area(figsize=(9, 4.5), ylim=(0, 1), title=f"Post-hoc temporal status · {cohort.replace('_', ' ')} · {TARGET_LABEL[target]}")
        plt.ylabel('Family proportion'); plt.xlabel('Temporal distance'); plt.tight_layout(); plt.show()

## 6. Quantitative temporal changes

These tables pair identical observational units before aggregation. They report adjacent and cumulative native-unit changes, paired support, improvement/deterioration fractions, 95% paired hierarchical-bootstrap intervals, and interval directionality. Prevalence and activation-magnitude ratios additionally quantify movement toward or away from 1.

In [ ]:
deltas = DERIVED['paired_temporal_deltas'].copy()
evidence = DERIVED['delta_evidence'].copy()
delta_columns = ['metric', 'delta_kind', 'cohort_view', 'matching_view', 'rule_source', 'activation_target', 'status', 'from_distance', 'to_distance', 'previous_mean', 'current_mean', 'delta', 'relative_change', 'stability_deviation_change', 'paired_support', 'fraction_negative', 'fraction_positive', 'lower_95', 'upper_95', 'ci_excludes_zero']
delta_columns = [column for column in delta_columns if column in deltas]
for metric in ('macro_f1', 'death_f1', 'f2', 'prevalence_ratio', 'jaccard', 'feature_association_cosine', 'activation_magnitude_ratio', 'tcav', 'status_proportion'):
    display(Markdown(f"### {metric.replace('_', ' ').title()} deltas"))
    display(deltas[deltas['metric'].eq(metric)][delta_columns].sort_values(['cohort_view', 'activation_target', 'delta_kind', 'to_distance'], na_position='last'))
display(Markdown('### Numeric evidence summary'))
display(evidence.sort_values(['metric', 'cohort_view', 'activation_target'], na_position='last'))

## 7. Relationship with model performance

The direct degradation, lead–lag, and change-point views below are exploratory associations, not causal evidence. Both sides of every association refer to the original fitted system; balanced-context sensitivity results are excluded.

In [ ]:
model_units = (performance.groupby(['reference_year', 'patient_split_seed', 'cohort_view', 'temporal_distance'], as_index=False)[['macro_f1', 'death_f1']].mean())
concept_units = (factors[(factors['matching_view'].eq('intersection')) & (factors['rule_source'].eq('semantic'))]
    .groupby(['reference_year', 'patient_split_seed', 'cohort_view', 'activation_target', 'temporal_distance'], as_index=False)[['f2', 'prevalence_instability', 'jaccard', 'feature_association_cosine', 'activation_magnitude_ratio']].mean())
aligned = model_units.merge(concept_units, on=['reference_year', 'patient_split_seed', 'cohort_view', 'temporal_distance'], how='inner')
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for axis, metric in zip(axes, ('prevalence_instability', 'jaccard', 'feature_association_cosine', 'activation_magnitude_ratio')):
    for target in ACTIVATION_TARGETS:
        selected = aligned[aligned['activation_target'].eq(target)]
        axis.scatter(selected[metric], selected['death_f1'], alpha=.45, label=TARGET_LABEL[target])
    axis.set(xlabel=metric.replace('_', ' '), ylabel='Death F1', title='Death F1 vs ' + metric.replace('_', ' '))
axes[-1].legend(); fig.tight_layout(); plt.show()

exploratory = []
for (cohort, target), group in aligned.groupby(['cohort_view', 'activation_target']):
    for lag in (0, 1, 2):
        shifted = group.copy()
        shifted['concept_distance'] = shifted['temporal_distance'] + lag
        paired = group[['reference_year', 'patient_split_seed', 'temporal_distance', 'death_f1']].merge(
            shifted[['reference_year', 'patient_split_seed', 'concept_distance', 'prevalence_instability', 'feature_association_cosine']],
            left_on=['reference_year', 'patient_split_seed', 'temporal_distance'], right_on=['reference_year', 'patient_split_seed', 'concept_distance'])
        for metric in ('prevalence_instability', 'feature_association_cosine'):
            exploratory.append({'cohort_view': cohort, 'activation_target': target, 'lag': lag, 'metric': metric, 'correlation': paired['death_f1'].corr(paired[metric]), 'paired_rows': len(paired)})
display(pd.DataFrame(exploratory))
display(Markdown('**Exploratory parent change-point diagnostics**'))
display(PARENT_TABLES['change_points'])

## 8. Balanced-context sensitivity experiment

Two consecutive distances with aggregate zero original-system death F1 trigger a deterministic equal-death/survivor reference-context fit. That re-contextualized model is a separate sensitivity experiment only: it never replaces the original-system performance in headline plots, temporal deltas, concept–performance associations, or CRI analyses. The table and plots below compare systems explicitly and must not be interpreted as a threshold-only sensitivity analysis.

In [ ]:
display(DERIVED['balanced_context_sensitivity_audit'])
variant_audit = (DERIVED['performance_variants'].groupby(['variant', 'cohort_view', 'temporal_distance'], as_index=False)
    .agg(macro_f1=('macro_f1', 'mean'), death_f1=('death_f1', 'mean'), rows=('reference_year', 'size'), deaths=('death_count', 'sum'), survivors=('survivor_count', 'sum')))
display(variant_audit)
for cohort in COHORTS:
    selected = variant_audit[variant_audit['cohort_view'].eq(cohort)]
    if selected.empty: continue
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
    for axis, (metric, label) in zip(axes, [('macro_f1', 'Macro F1'), ('death_f1', 'Death F1')]):
        for variant, rows in selected.groupby('variant'):
            rows = rows.sort_values('temporal_distance')
            axis.plot(rows['temporal_distance'], rows[metric], marker='o', label=variant.replace('_', ' '))
        axis.set(title=label, xlabel='Temporal distance', ylim=(0, 1))
    axes[0].set_ylabel('F1')
    axes[-1].legend()
    fig.suptitle(f"Separate system sensitivity · {cohort.replace('_', ' ')}")
    fig.tight_layout(); plt.show()

## 9. Reproducibility and invariant checks

In [ ]:
assert set(factors['cohort_view'].dropna()) <= set(COHORTS)
assert set(factors['matching_view'].dropna()) <= {'cosine_qualified', 'intersection'}
assert set(np.round(factors['activation_target'].dropna().astype(float), 2)) <= set(ACTIVATION_TARGETS)
assert factors['geometric_factor_recurrence'].astype(float).gt(RECURRENCE_MIN).all()
assert ladder['matched'].le(ladder['eligible']).all()
assert ladder['recurrent'].le(ladder['matched']).all()
assert ladder['consensus'].le(ladder['recurrent']).all()
assert ladder['interpretable'].le(ladder['consensus']).all()
assert ladder['dual_source'].le(ladder['interpretable']).all()
assert not deltas['metric'].eq('tcav_normalized').any()
assert performance['selected_variant'].eq('original').all()
original_performance = DERIVED['performance_variants'][DERIVED['performance_variants']['variant'].eq('original')]
performance_identity = ['reference_year', 'patient_split_seed', 'test_year', 'temporal_distance', 'cohort_view']
performance_values = performance_identity + ['macro_f1', 'death_f1']
pd.testing.assert_frame_equal(
    performance[performance_values].sort_values(performance_identity).reset_index(drop=True),
    original_performance[performance_values].sort_values(performance_identity).reset_index(drop=True),
    check_dtype=False,
)
assert ENRICHMENT_MANIFEST['parent_hash'] == PARENT_HASH
display(pd.DataFrame({
    'invariant': ['complete parent', 'pinned identity', 'two cohorts only', 'P90/P70/P50 remain explicit', 'strict recurrence', 'nested ladder', 'actual TCAV only', 'original performance only', 'concept-performance system coupling'],
    'passed': [True] * 9,
}))
display(Markdown(f"Derived artifact manifest: `{ENRICHMENT_MANIFEST_PATH}`"))

## Conceptual Robustness Index

CRI is derived from frozen d0 semantic/intersection families. It keeps `all_comer` and `pipeline_unseen`, and P90/P70/P50, separate. Coverage is reported beside every score; missing technical support is not treated as concept collapse.

In [ ]:
cri_system = CRI['cri_system_summaries'].copy()
cri_oof = CRI['cri_loyo_metrics'].copy()
display(cri_system.sort_values(['cohort_view', 'activation_target', 'reference_year', 'temporal_distance']))
display(cri_oof.sort_values(['cohort_view', 'activation_target', 'model']))

fig, axes = plt.subplots(len(COHORTS), len(ACTIVATION_TARGETS), figsize=(14, 6), sharex=True, sharey=True)
for i, cohort in enumerate(COHORTS):
    for j, target in enumerate(ACTIVATION_TARGETS):
        axis = axes[i, j]
        rows = cri_system[(cri_system.cohort_view == cohort) & (cri_system.activation_target == target)]
        summary = rows.groupby('temporal_distance', as_index=False).agg(median_cri=('median_cri', 'median'), coverage=('coverage', 'median'))
        axis.plot(summary.temporal_distance, summary.median_cri, marker='o', label='median CRI')
        axis.plot(summary.temporal_distance, summary.coverage, marker='s', linestyle='--', label='coverage')
        axis.set_title(f'{cohort} / {TARGET_LABEL[target]}')
        axis.set_ylim(0, 1); axis.legend(fontsize=8)
fig.suptitle('CRI and coverage remain separate estimands')
fig.tight_layout()
display(Markdown(f"CRI artifact manifest: `{CRI_MANIFEST_PATH}`"))

In [ ]:
cri_oof